# Agent 3: Deep Researcher / Roadmap Generator Agent

This notebook implements the complete, detailed implementation of the **Deep Researcher / Roadmap Generator Agent** used in CareerAtlas. The agent coordinates an iterative web search and planning loop using LangGraph, structures the gathered links into Phase milestones, runs grounding and HTTP liveness checks to drop dead links, and grades the pathway against a strict evaluation rubric.

### Step 1: API Keys Setup
Please configure your API keys here.

In [ ]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API Key: ")

class Settings:
    groq_api_key = os.environ.get("GROQ_API_KEY", "")
    groq_model = "llama-3.3-70b-versatile"
    tavily_api_key = os.environ.get("TAVILY_API_KEY", "")

settings = Settings()

In [ ]:
# Install required dependencies
# !pip install pydantic langgraph langchain-groq tavily-python httpx

### Step 2: Imports & Pydantic Schemas
These define the LangGraph state and output structures.

In [ ]:
import asyncio
import logging
from datetime import datetime
from typing import List, Literal, Optional, TypedDict, Tuple
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from tavily import TavilyClient
import httpx

class Resource(BaseModel):
    title: str
    kind: str = Field(description="course | doc | video | article | book")
    provider: str = Field(description="e.g. Coursera, official docs")
    url: str
    why: str = Field(description="1 sentence reasoning")

class Milestone(BaseModel):
    phase: Literal["Foundations", "Intermediate", "Advanced"]
    skill: str
    estimated_weeks: int
    objective: str
    resources: List[Resource] = Field(default_factory=list)
    checklist: List[str]
    mini_project: Optional[str] = None

class Pathway(BaseModel):
    target_role: str
    milestones: List[Milestone]
    rationale: str

class GapIn(BaseModel):
    skill: str
    category: str
    relevance: int
    difficulty: str
    prerequisites: List[str] = Field(default_factory=list)

class NextQuery(BaseModel):
    query: str
    rationale: str

class CriticOut(BaseModel):
    decision: Literal["continue", "structure"]
    rationale: str

class JudgeScore(BaseModel):
    criterion: Literal["coverage", "ordering", "resource_quality", "actionability", "personalization", "grounding", "recency"]
    score: int
    rationale: str

class JudgeVerdict(BaseModel):
    overall_score: float = Field(ge=1, le=5)
    pass_fail: Literal["pass", "fail"]
    strengths: List[str] = Field(default_factory=list)
    weaknesses: List[str] = Field(default_factory=list)
    improvement_actions: List[str] = Field(default_factory=list)
    rubric_scores: List[JudgeScore] = Field(default_factory=list)

class DroppedResource(BaseModel):
    milestone_skill: str
    title: str
    url: str
    reason: Literal["ungrounded", "dead_link"]

class ValidationResult(BaseModel):
    checked: int = 0
    kept: int = 0
    dropped: List[DroppedResource] = Field(default_factory=list)

class ResearcherState(TypedDict, total=False):
    gaps: List[GapIn]
    target_role: str
    notes: List[str]
    last_query: str
    iteration: int
    max_iter: int
    pathway: Pathway
    valid_urls: List[str]
    validation: dict
    judge_verdict: dict
    retry_count: int
    max_retry: int

### Step 3: Prompts & Search Helper

In [ ]:
PLAN_PROMPT = ChatPromptTemplate.from_template("""You are a research planner.
TARGET ROLE: {target_role}
OUTSTANDING GAPS:
{gaps}
NOTES GATHERED:
{notes}
Propose the next single search query.
""")

CRITIC_PROMPT = ChatPromptTemplate.from_template("""Decide if notes are enough to structure the roadmap.
NOTES:
{notes}
Option: continue or structure.
""")

STRUCTURE_PROMPT = ChatPromptTemplate.from_template("""Build a pathway using ONLY links in the research notes.
GAPS:
{gaps}
NOTES:
{notes}
""")

def search_web(query: str) -> List[dict]:
    try:
        client = TavilyClient(api_key=settings.tavily_api_key)
        raw = client.search(query, max_results=3, time_range="year")
        return raw.get("results", []) or []
    except Exception as e:
        print(f"Tavily search failed: {e}")
        return []

### Step 4: Grounding & Liveness HTTP Validation
Exact async URL probing and grounding validator from `app.deep_researcher.validation`.

In [ ]:
_UA = "Mozilla/5.0 (compatible; CareerAtlas/1.0)"
_RETRY_GET_STATUSES = {401, 403, 405, 429}

def _norm(url: str) -> str:
    return (url or "").strip().rstrip("/").lower()

async def _probe(client: httpx.AsyncClient, url: str) -> Tuple[str, bool]:
    try:
        r = await client.head(url, follow_redirects=True, timeout=6.0)
        if r.status_code in _RETRY_GET_STATUSES:
            r = await client.get(url, follow_redirects=True, timeout=8.0)
        return url, r.status_code >= 400
    except Exception:
        return url, False

async def _probe_all(urls: List[str]) -> dict:
    if not urls: return {}
    async with httpx.AsyncClient(headers={"User-Agent": _UA}) as client:
        results = await asyncio.gather(*[_probe(client, u) for u in urls])
    return {u: dead for u, dead in results}

def validate_pathway(pathway: Pathway, valid_urls: List[str]) -> Tuple[Pathway, ValidationResult]:
    grounded_set = {_norm(u) for u in (valid_urls or []) if u}
    all_urls = []
    for m in pathway.milestones:
        for r in m.resources:
            if r.url: all_urls.append(r.url)
            
    grounded_urls = [u for u in all_urls if _norm(u) in grounded_set]
    dead_map = asyncio.run(_probe_all(list(set(grounded_urls))))
    
    dropped = []
    kept = 0
    new_milestones = []
    for m in pathway.milestones:
        surviving = []
        for r in m.resources:
            norm = _norm(r.url)
            if norm not in grounded_set:
                dropped.append(DroppedResource(milestone_skill=m.skill, title=r.title, url=r.url, reason="ungrounded"))
                continue
            if dead_map.get(r.url, False):
                dropped.append(DroppedResource(milestone_skill=m.skill, title=r.title, url=r.url, reason="dead_link"))
                continue
            surviving.append(r)
            kept += 1
        new_milestones.append(m.model_copy(update={"resources": surviving}))
        
    cleaned = pathway.model_copy(update={"milestones": new_milestones})
    result = ValidationResult(checked=len(all_urls), kept=kept, dropped=dropped)
    return cleaned, result

### Step 5: Judge Evaluator & LangGraph Nodes
Evaluates pathway quality via Groq and binds everything into StateGraph.

In [ ]:
JUDGE_PROMPT = ChatPromptTemplate.from_template("""You are a strict evaluator.
Score the pathway 1-5.
GAPS: {gaps}
NOTES: {notes}
VALIDATION: {validation}
PATHWAY JSON:
{pathway_json}
""")

def evaluate_pathway(target_role: str, gaps: List[GapIn], notes: List[str], pathway: Pathway, validation: ValidationResult, current_year: int) -> JudgeVerdict:
    model = ChatGroq(model=settings.groq_model, groq_api_key=settings.groq_api_key, temperature=0.0)
    judge = JUDGE_PROMPT | model.with_structured_output(JudgeVerdict)
    gaps_text = "\n".join(f"- {g.skill}" for g in gaps)
    validation_text = f"{validation.kept}/{validation.checked} kept"
    return judge.invoke({
        "gaps": gaps_text,
        "notes": "\n".join(notes),
        "validation": validation_text,
        "pathway_json": pathway.model_dump_json(indent=2)
    })

def node_plan(state: ResearcherState) -> dict:
    model = ChatGroq(model=settings.groq_model, groq_api_key=settings.groq_api_key, temperature=0.2)
    planner = PLAN_PROMPT | model.with_structured_output(NextQuery)
    nxt = planner.invoke({"target_role": state["target_role"], "gaps": "\n".join(f"- {g.skill}" for g in state["gaps"]), "notes": "\n".join(state.get("notes", []))})
    return {"last_query": nxt.query, "iteration": state.get("iteration", 0) + 1}

def node_search(state: ResearcherState) -> dict:
    results = search_web(state["last_query"])
    lines, urls = [], []
    for r in results:
        url = r.get("url", "")
        lines.append(f"{url} :: {r.get('content', '')[:300]}")
        if url: urls.append(url)
    return {"notes": state.get("notes", []) + ["\n".join(lines)], "valid_urls": state.get("valid_urls", []) + urls}

def critic_route(state: ResearcherState) -> str:
    if state.get("iteration", 0) >= state.get("max_iter", 2): return "structure"
    model = ChatGroq(model=settings.groq_model, groq_api_key=settings.groq_api_key, temperature=0.2)
    critic = CRITIC_PROMPT | model.with_structured_output(CriticOut)
    out = critic.invoke({"notes": "\n".join(state.get("notes", []))})
    return out.decision

def node_structure(state: ResearcherState) -> dict:
    model = ChatGroq(model=settings.groq_model, groq_api_key=settings.groq_api_key, temperature=0.2)
    structurer = STRUCTURE_PROMPT | model.with_structured_output(Pathway)
    pathway = structurer.invoke({"target_role": state["target_role"], "gaps": "\n".join(f"- {g.skill}" for g in state["gaps"]), "notes": "\n".join(state.get("notes", []))})
    return {"pathway": pathway}

def node_validate(state: ResearcherState) -> dict:
    pathway = state["pathway"]
    cleaned, result = validate_pathway(pathway, state.get("valid_urls", []))
    return {"pathway": cleaned, "validation": result.model_dump()}

g = StateGraph(ResearcherState)
g.add_node("plan", node_plan)
g.add_node("search", node_search)
g.add_node("structure", node_structure)
g.add_node("validate", node_validate)
g.add_edge(START, "plan")
g.add_edge("plan", "search")
g.add_conditional_edges("search", critic_route, {"continue": "plan", "structure": "structure"})
g.add_edge("structure", "validate")
g.add_edge("validate", END)
deep_researcher_agent = g.compile()

### Step 6: Milestone Syncing & Persistence
Status-preserving milestone upsert matching `app.roadmap_generation.service`.

In [ ]:
class MockResponse:
    def __init__(self, data):
        self.data = data

class MockQuery:
    def __init__(self, data=None):
        self._data = data or []
    def select(self, *args, **kwargs): return self
    def eq(self, *args, **kwargs): return self
    def execute(self): return MockResponse(self._data)

class MockTable:
    def __init__(self, data=None):
        self._data = data or []
    def select(self, *args, **kwargs): return MockQuery(self._data)
    def delete(self, *args, **kwargs): return MockQuery()
    def upsert(self, *args, **kwargs): return MockQuery()

class MockDbClient:
    def __init__(self, initial_data=None):
        self._data = initial_data or {}
    def table(self, name):
        return MockTable(self._data.get(name, []))

def upsert_role_milestones(user_id: str, role_id: str, role_title: str, resume_id: str | None, milestones: List[dict], db_client_mock: Any) -> List[dict]:
    existing = db_client_mock.table("milestones").select("id,skill,status,completed_at").eq("user_id", user_id).eq("target_role_id", role_id).execute()
    prior_by_skill = {(row.get("skill") or "").strip().lower(): row for row in existing.data if row.get("skill")}
    
    rows_to_upsert = []
    for idx, ms in enumerate(milestones):
        key = (ms.get("skill") or "").strip().lower()
        prior = prior_by_skill.get(key)
        if prior:
            status = prior.get("status") or "locked"
            completed_at = prior.get("completed_at")
        else:
            status = "in_progress" if idx == 0 else "locked"
            completed_at = None
        row = {**ms, "status": status, "completed_at": completed_at, "sort_order": idx}
        if prior and prior.get("id"): row["id"] = prior["id"]
        rows_to_upsert.append(row)
    return rows_to_upsert

### Step 7: Test Run

In [ ]:
gaps_in = [GapIn(skill="PyTorch", category="framework", relevance=90, difficulty="Medium")]
state_input = {"gaps": gaps_in, "target_role": "Machine Learning Engineer", "notes": [], "iteration": 0, "max_iter": 2, "valid_urls": []}

try:
    final_state = deep_researcher_agent.invoke(state_input, {"recursion_limit": 20})
    print("Generated Pathway:")
    print(final_state["pathway"].model_dump_json(indent=2))
    
    # Test milestone persistence locally with mock client
    db_mock = MockDbClient({"milestones": [{"skill": "PyTorch", "status": "completed", "completed_at": "2026-07-06"}]})
    new_ms = [{"skill": "PyTorch", "objective": "ML"}, {"skill": "Docker", "objective": "Ops"}]
    synced = upsert_role_milestones("user-1", "role-1", "ML Engineer", "resume-1", new_ms, db_mock)
    print("\nSynced Milestones:")
    print(json.dumps(synced, indent=2))
except Exception as e:
    print(f"Execution skipped or failed. Error: {e}")